In [11]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np
import time

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os  

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

from requests.exceptions import ConnectTimeout, ReadTimeout, RequestException

In [2]:
date = "2025-11"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

In [3]:
end_use = "naics?get=CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url) 
    
print(r)
    
df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

df.columns = r.json()[0]

df["total_imports"] = df["CON_VAL_MO"].astype(float)

df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])

top_products = grp.agg({"total_imports":"sum","CTY_CODE":"first"})

country_list = list(top_products.sort_values(by = "total_imports", ascending = False).CTY_CODE)[0:31]


['TOTAL FOR ALL COUNTRIES','NAFTA','EUROPEAN UNION']

<Response [200]>


['TOTAL FOR ALL COUNTRIES', 'NAFTA', 'EUROPEAN UNION']

In [4]:
df.tail()

,CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL,time,total_imports
39049,13896714,7940,ZAMBIA,DET,2025-11,13896714.0
39050,12624834,7950,ESWATINI,DET,2025-11,12624834.0
39051,15520151,7960,ZIMBABWE,DET,2025-11,15520151.0
39052,2368430,7970,MALAWI,DET,2025-11,2368430.0
39053,8270767,7990,LESOTHO,DET,2025-11,8270767.0


In [5]:
country_list[0] = ""

In [6]:
country_list.extend(["0003", "0020"])

In [7]:
len(country_list)

33

In [12]:
end_use = "hs?get=CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

surl  = surl + my_key + "&COMM_LVL=HS10" 

for xxx in country_list:
    
    out_file = ".\\data"+ "\\imports-hs10\\" + xxx + "data-" + date + ".parquet"
    
    if xxx == "":
        out_file = ".\\data"+ "\\imports-hs10\\" + "TOTAL" + "data-" + date + ".parquet"
    
    
    if os.path.exists(out_file):
        
        print("Already have downloaded file")
        
        continue
    
    print(f"Downloading {xxx} for {date}")
    
    url = surl + "&time=" + date
    
    if xxx != "":
        url = url + "&CTY_CODE=" + xxx
    
    max_retries = 5
    retry_count = 0
    r = None
    
    while retry_count < max_retries:
        try:
            # Added timeout (30 seconds) and read timeout
            r = requests.get(url, timeout=30)
            
            if r.status_code == 200:
                break
            else:
                print(f"Request failed with status {r.status_code}, waiting 30 seconds...")
                time.sleep(30)
                retry_count += 1
                
        except (ConnectTimeout, ReadTimeout) as e:
            retry_count += 1
            print(f"Connection timeout on attempt {retry_count}/{max_retries}: {type(e).__name__}")
            if retry_count < max_retries:
                wait_time = 30 * (2 ** (retry_count - 1))  # Exponential backoff
                print(f"Waiting {wait_time} seconds before retry...")
                time.sleep(wait_time)
            else:
                print(f"Max retries exceeded for {xxx}, skipping...")
                continue
                
        except RequestException as e:
            print(f"Request failed with error: {e}")
            retry_count += 1
            if retry_count < max_retries:
                time.sleep(30)
    
    if r is None or r.status_code != 200:
        print(f"Failed to download {xxx} after {max_retries} attempts, skipping...")
        continue
    
    print(r)
    
    foo = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

    foo.columns = r.json()[0]

    pq.write_table(pa.Table.from_pandas(foo), out_file)


Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
<Response [200]>
<Response [200]>
<Response [200]>


In [13]:
# Append dated files to current files and replace them
import glob

data_dir = ".\\data\\imports-hs10"

# Find all current files
current_files = glob.glob(os.path.join(data_dir, "*data-current.parquet"))

for current_file in current_files:
    # Extract the country code from the filename
    filename = os.path.basename(current_file)
    country_code = filename.replace("data-current.parquet", "")
    
    # Find the corresponding dated file
    dated_file = os.path.join(data_dir, f"{country_code}data-{date}.parquet")
    
    if not os.path.exists(dated_file):
        print(f"Dated file not found for {country_code}, skipping...")
        continue
    
    print(f"Processing {country_code}...")
    
    # Read both files
    current_df = pq.read_table(current_file).to_pandas()
    dated_df = pq.read_table(dated_file).to_pandas()
    
    # Append dated data to current
    combined_df = pd.concat([current_df, dated_df], ignore_index=True)
    
    # Write back to current file
    pq.write_table(pa.Table.from_pandas(combined_df), current_file)
    
    print(f"Updated {country_code}: appended {len(dated_df)} rows, total now {len(combined_df)}")

print("Done!")


Processing 0003...
Updated 0003: appended 15888 rows, total now 2208642
Processing 0020...
Updated 0020: appended 13826 rows, total now 1849986
Processing 1220...
Updated 1220: appended 12073 rows, total now 1568378
Processing 2010...
Updated 2010: appended 9987 rows, total now 1290154
Processing 3010...
Updated 3010: appended 3903 rows, total now 388161
Processing 3370...
Updated 3370: appended 1757 rows, total now 188126
Processing 3510...
Updated 3510: appended 5928 rows, total now 682817
Processing 4010...
Updated 4010: appended 4843 rows, total now 584165
Processing 4120...
Updated 4120: appended 10357 rows, total now 1314866
Processing 4190...
Updated 4190: appended 2967 rows, total now 327829
Processing 4210...
Updated 4210: appended 6827 rows, total now 811620
Processing 4231...
Updated 4231: appended 5501 rows, total now 663454
Processing 4279...
Updated 4279: appended 9668 rows, total now 1229677
Processing 4280...
Updated 4280: appended 10809 rows, total now 1452561
Processi

In [15]:
combined_df.head()

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC,time,COMM_LVL
0,TOTAL FOR ALL COUNTRIES,773010,0,0602400000,"ROSES, GRAFTED OR NOT",2013-01,HS10
1,TOTAL FOR ALL COUNTRIES,6177543,0,0602902000,"ORCHID PLANTS, LIVE",2013-01,HS10
2,TOTAL FOR ALL COUNTRIES,135786,0,0602903010,CHRYSANTHEMUMS WITH SOIL ATTACHED TO ROOTS,2013-01,HS10
3,TOTAL FOR ALL COUNTRIES,169439,0,0602903090,"HERBACEOUS PERENNIALS,WTH SOIL ATTACHED,LIVE,N...",2013-01,HS10
4,TOTAL FOR ALL COUNTRIES,2177498,25345,0602904000,HERBACEOUS PERENNIALS WTHOUT SOIL ATTACHED NESOI,2013-01,HS10
